# Streamlit 기초 - 데이터 대시보드 만들기

| 구분 | Flask/Django | Streamlit |
|---|---|---|
| 목적 | 범용 웹 앱 | 데이터/ML 대시보드 특화 |
| 코드량 | 많음 (HTML, JS 필요) | **적음 (순수 Python)** |
| 학습 곡선 | 높음 | **낮음 (1~2시간)** |
| 재실행 | 수동 라우팅 | **위젯 변경 시 자동 재실행** |

```bash
pip install streamlit plotly pandas
```

## 1. 일단 시작해보기

In [ ]:
%streamlit hello.py

UsageError: Line magic function `%streamlit` not found.


---
## 2. 핵심 위젯 알아보기

| 위젯 함수 | 설명 | 반환 타입 |
|---|---|---|
| `st.slider()` | 숫자 범위 슬라이더 | `int` / `float` |
| `st.selectbox()` | 드롭다운 선택 | `str` |
| `st.multiselect()` | 다중 선택 | `list` |
| `st.checkbox()` | 체크박스 | `bool` |
| `st.radio()` | 라디오 버튼 | `str` |
| `st.text_input()` | 텍스트 입력 | `str` |
| `st.number_input()` | 숫자 입력 | `int` / `float` |
| `st.file_uploader()` | 파일 업로드 | `UploadedFile` |
| `st.button()` | 클릭 버튼 | `bool` |

In [ ]:
# 위젯 예제 앱 생성
widget_code = '''import streamlit as st
import pandas as pd
import numpy as np

st.title("Streamlit 위젯 갤러리")

# ── 사이드바에 위젯 배치 ──────────────────────────
with st.sidebar:
    st.header("⚙️ 설정")
    n_points = st.slider("데이터 수", 20, 300, 100)
    chart_type = st.selectbox("차트 종류", ["Line", "Bar", "Area"])
    show_raw  = st.checkbox("원본 데이터 표시", value=True)
    selected_cols = st.multiselect("표시할 열", ["A","B","C"], default=["A","B"])

# ── 본문 ─────────────────────────────────────────
df = pd.DataFrame(np.random.randn(n_points, 3), columns=["A","B","C"])
plot_df = df[selected_cols] if selected_cols else df

st.subheader(f"{chart_type} Chart")
if chart_type == "Line":  st.line_chart(plot_df)
elif chart_type == "Bar": st.bar_chart(plot_df)
else:                     st.area_chart(plot_df)

if show_raw:
    st.subheader("원본 데이터")
    st.dataframe(df.describe().round(3))
'''
with open('widget_gallery.py', 'w', encoding='utf-8') as f:
    f.write(widget_code)

---
## 3. Plotly + Streamlit: 인터랙티브 차트

Streamlit 기본 차트보다 **Plotly**가 훨씬 강력합니다:
- 줌, 팬, 호버 툴팁 기본 제공
- 클릭으로 범례 토글
- 애니메이션, 3D 플롯 지원
- `st.plotly_chart(fig, use_container_width=True)`로 삽입

In [ ]:
# Plotly 차트 종류 미리보기
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# 샘플 데이터
np.random.seed(42)
df_demo = pd.DataFrame({
    'x': range(50),
    'y': np.cumsum(np.random.randn(50)),
    'category': np.random.choice(['A', 'B', 'C'], 50),
    'size': np.random.randint(5, 30, 50)
})

# 서브플롯으로 여러 차트 한 번에 보기
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Line Chart (hover 해보세요)',
        'Scatter with Color/Size',
        'Box Plot by Category',
        'Histogram'
    )
)

fig.add_trace(go.Scatter(x=df_demo['x'], y=df_demo['y'], mode='lines+markers',
                         line=dict(color='#4ECDC4', width=2), name='Trend'), row=1, col=1)
for cat, color in zip(['A','B','C'], ['#FF6B6B','#45B7D1','#FFE66D']):
    mask = df_demo['category'] == cat
    fig.add_trace(go.Scatter(x=df_demo.loc[mask,'x'], y=df_demo.loc[mask,'y'],
                             mode='markers', name=f'Cat {cat}',
                             marker=dict(color=color, size=df_demo.loc[mask,'size'])), row=1, col=2)
for cat, color in zip(['A','B','C'], ['#FF6B6B','#45B7D1','#FFE66D']):
    mask = df_demo['category'] == cat
    fig.add_trace(go.Box(y=df_demo.loc[mask,'y'], name=f'Cat {cat}',
                         marker_color=color), row=2, col=1)
fig.add_trace(go.Histogram(x=df_demo['y'], nbinsx=20,
                           marker_color='#96CEB4', name='Dist'), row=2, col=2)

fig.update_layout(height=600, title_text='Plotly 인터랙티브 차트 예시',
                  template='plotly_white', showlegend=False)
fig.show()

---
## 4. 레이아웃: 칼럼 & 탭

```python
# 칼럼 분할
col1, col2, col3 = st.columns([2, 1, 1])   # 비율 지정 가능
with col1:
    st.plotly_chart(fig1)
with col2:
    st.metric("정확도", "92.3%", "+1.2%")  # KPI 카드

# 탭 분할
tab1, tab2 = st.tabs(["📊 차트", "📋 데이터"])
with tab1:
    st.plotly_chart(fig)
with tab2:
    st.dataframe(df)
```

---
## 5. 캐싱 (@st.cache_data)

> **문제**: Streamlit은 위젯 변경 시 전체 스크립트를 재실행 → 매번 CSV를 읽으면 느림  
> **해결**: `@st.cache_data` 데코레이터로 결과를 캐시

```python
@st.cache_data          # 첫 호출 결과를 메모리에 저장
def load_data(path):
    return pd.read_csv(path)  # 두 번째 호출부터는 캐시에서 즉시 반환

df = load_data('../data/train.csv')   # 빠름!
```

---
## 6. 미니 대시보드 완성 앱

지금까지 배운 내용을 모두 합친 **Mini Dashboard**를 생성합니다.  
`mini_dashboard.py`를 실행해 보세요!

---
## 7. Streamlit 핵심 명령어 치트시트

```python
# 텍스트/제목
st.title(), st.header(), st.subheader(), st.markdown(), st.write()

# 데이터 표시
st.dataframe(df)             # 정렬/필터 가능한 인터랙티브 표
st.table(df)                 # 정적 표
st.metric("KPI", 92.3, 1.2)  # KPI 카드 (▲/▼ 색상 자동)
st.json(dict)                # JSON 뷰어

# 차트
st.plotly_chart(fig, use_container_width=True)
st.line_chart(df) / st.bar_chart(df) / st.area_chart(df)  # 간단 차트

# 레이아웃
col1, col2 = st.columns([3, 1])
tab1, tab2 = st.tabs(["이름1", "이름2"])
with st.sidebar: ...
with st.expander("자세히 보기"): ...

# 상태/피드백
st.success("완료!") / st.error("오류") / st.warning("주의") / st.info("정보")
with st.spinner("로딩 중..."):
    ...

# 캐싱
@st.cache_data    # 데이터/계산 결과 캐시
@st.cache_resource  # 모델/DB 연결 캐시 (전역 공유)
```
